In [ ]:
!pip install pyspark -q
from google.colab import drive
drive.mount('/content/drive')

import os, datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

BASE_PATH = "/content/drive/MyDrive/HM-DATA/"
INPUT_TRANS = BASE_PATH + "processed_v2/cleaned_transactions.parquet"
OUTPUT_DIR = BASE_PATH + "outputs_v2/candidates/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("1. Khoi tao Spark Session cho ItemCF (Validation Mode)...")
spark = SparkSession.builder \
    .appName("Retrieval_True_ItemCF_Val") \
    .config("spark.driver.memory", "10g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

Mounted at /content/drive
1. Khoi tao Spark Session cho ItemCF (Validation Mode)...


In [ ]:
print("2. Doc du lieu va chia khung thoi gian (Validation - 42 ngay)...")
transactions = spark.read.parquet(INPUT_TRANS)
max_date = transactions.select(F.max("t_dat_date")).collect()[0][0]

test_start = max_date - datetime.timedelta(days=7)
val_start = test_start - datetime.timedelta(days=7)

# Cua so 42 ngay (6 tuan)
train_hist_start = val_start - datetime.timedelta(days=42)
test_hist_start = test_start - datetime.timedelta(days=42)

train_hist_df = transactions.filter((F.col("t_dat_date") >= train_hist_start) & (F.col("t_dat_date") < val_start))
test_hist_df = transactions.filter((F.col("t_dat_date") >= test_hist_start) & (F.col("t_dat_date") < test_start))

print(f"Train History range: {train_hist_start} to {val_start}")
print(f"Test History range:  {test_hist_start} to {test_start}")

2. Doc du lieu va chia khung thoi gian (Validation - 42 ngay)...
Train History range: 2020-07-28 to 2020-09-08
Test History range:  2020-08-04 to 2020-09-15


In [ ]:
def generate_itemcf_candidates(history_df, top_n=20):
    # Buoc 1: Loc du lieu mua hang duy nhat (1 user mua 1 item nhieu lan tinh la 1)
    user_item = history_df.select("customer_id", "article_id").dropDuplicates()

    # Buoc 2: Tao ma tran dong xuat hien (Self-Join)
    pair_df = user_item.alias("i1").join(
        user_item.alias("i2"),
        F.col("i1.customer_id") == F.col("i2.customer_id")
    ).filter(F.col("i1.article_id") < F.col("i2.article_id"))

    # Dem so lan 2 item duoc mua cung nhau
    co_occur = pair_df.groupBy(
        F.col("i1.article_id").alias("item_A"),
        F.col("i2.article_id").alias("item_B")
    ).agg(F.count("*").alias("score"))

    # Nhan doi ma tran de dam bao tinh hai chieu
    co_occur_sym = co_occur.select(F.col("item_A").alias("item1"), F.col("item_B").alias("item2"), "score") \
        .union(co_occur.select(F.col("item_B").alias("item1"), F.col("item_A").alias("item2"), "score"))

    # Lay Top 20 san pham tuong dong nhat cho moi san pham
    window_item = Window.partitionBy("item1").orderBy(F.col("score").desc())
    sim_items = co_occur_sym.withColumn("rn", F.row_number().over(window_item)) \
        .filter(F.col("rn") <= 20).drop("rn")

    # Buoc 3: De xuat ung vien cho khach hang (Tinh tong diem ItemCF)
    candidates = user_item.withColumnRenamed("article_id", "item1") \
        .join(sim_items, "item1") \
        .groupBy("customer_id", "item2") \
        .agg(F.sum("score").alias("itemcf_score_raw"))

    # Buoc 4: Xep hang va cat Top N
    window_user = Window.partitionBy("customer_id").orderBy(F.col("itemcf_score_raw").desc())
    top_cands = candidates.withColumn("rn", F.row_number().over(window_user)) \
        .filter(F.col("rn") <= top_n)

    # Buoc 5: Chuan hoa diem so (Min-Max Scaling) ve muc [0, 1] cho moi khach hang
    window_scale = Window.partitionBy("customer_id")
    final_cands = top_cands.withColumn("max_score", F.max("itemcf_score_raw").over(window_scale)) \
        .withColumn("min_score", F.min("itemcf_score_raw").over(window_scale)) \
        .withColumn("itemcf_score",
            F.when(F.col("max_score") == F.col("min_score"), 1.0)
             .otherwise((F.col("itemcf_score_raw") - F.col("min_score")) / (F.col("max_score") - F.col("min_score")))
        ) \
        .select("customer_id", F.col("item2").alias("article_id"), "itemcf_score") \
        .withColumn("strategy", F.lit("itemcf"))

    return final_cands

def evaluate_recall(candidates_df, target_df, target_start, target_end):
    actuals = target_df.filter((F.col("t_dat_date") >= target_start) & (F.col("t_dat_date") < target_end)) \
        .select("customer_id", "article_id").dropDuplicates()

    total_actuals = actuals.count()
    hits = actuals.join(candidates_df, ["customer_id", "article_id"], "inner").dropDuplicates().count()
    recall = hits / total_actuals if total_actuals > 0 else 0

    print(f"   -> Actuals: {total_actuals} | Hits: {hits} | Recall: {recall:.4f}")

In [ ]:
print("\n3. Generating True ItemCF candidates for Train set (Co diem itemcf_score)...")
train_cands = generate_itemcf_candidates(train_hist_df, top_n=20)
train_cands.write.mode("overwrite").parquet(OUTPUT_DIR + "train_itemcf.parquet")

print("4. Generating True ItemCF candidates for Test set (Co diem itemcf_score)...")
test_cands = generate_itemcf_candidates(test_hist_df, top_n=20)
test_cands.write.mode("overwrite").parquet(OUTPUT_DIR + "test_itemcf.parquet")

print("5. Evaluating TEST set:")
evaluate_recall(test_cands, transactions, test_start, max_date)
print("Hoan tat! Thu vien ItemCF (Phien ban nang cap diem so) da san sang.")


3. Generating True ItemCF candidates for Train set (Co diem itemcf_score)...
4. Generating True ItemCF candidates for Test set (Co diem itemcf_score)...
5. Evaluating TEST set:
   -> Actuals: 207996 | Hits: 5626 | Recall: 0.0270
Hoan tat! Thu vien ItemCF (Phien ban nang cap diem so) da san sang.
